# Setup

In [70]:
# !pip install wandb --quiet

# import wandb
# import warnings
# from kaggle_secrets import UserSecretsClient
# warnings.filterwarnings("ignore")

# user_secrets = UserSecretsClient()
# wandb_api = user_secrets.get_secret("WANDB_API_KEY")

# wandb.login(key=wandb_api)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [71]:
# run = wandb.init(
#     entity="23f2002974-dl-genai-project",
#     project="23f2002974-t12026",
#     name="Scratch_CNN_run3",
#     config={
#         "architecture": "Scratch CNN",
#         "epochs": 20,
#     }
# )

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▇██▅▂▃▃▁▂▂▁▃▂▃▃▁▁▂▂▁
train_acc,▁▅▆▆▇▇▇▇▇▇██████████
train_f1,▁▅▆▆▇▇▇▇▇▇██████████
train_loss,█▅▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▃▁▁▄▇▆▆█▇▇█▆▇▇▆██▇▇█
val_f1,▃▁▂▄▇▆▇█▇▇█▇▇▇▇██▇▇█
val_loss,▆██▅▂▃▂▁▂▂▁▃▂▂▃▁▁▂▂▁
epoch,20
lr,0.0003
train_acc,0.95616


# Libraries and imports

In [1]:
import os
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

import librosa
import torchaudio.transforms as T

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler


from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Data Pre-processing & Feature Extraction

In [5]:
"""Config"""
# Standard audio settings
SR = 22050
N_MELS = 128
DURATION = 30 

# Where the data is coming from
GENRE_INPUT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_INPUT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

# Where we want to save the processed files
GENRE_OUTPUT = "/kaggle/working/processed_mels"
NOISE_OUTPUT = "/kaggle/working/processed_noise_mels"

STEM_FILES = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]

In [6]:
def preprocess_dataset():
    
    '''Create the main output directory'''
    os.makedirs(GENRE_OUTPUT, exist_ok=True)

    all_items = os.listdir(GENRE_INPUT)
    genres = []

    for item in all_items:
        if os.path.isdir(os.path.join(GENRE_INPUT, item)):
            genres.append(item)

    for genre in genres:
        genre_input_path = os.path.join(GENRE_INPUT, genre)
        genre_output_path = os.path.join(GENRE_OUTPUT, genre)
        os.makedirs(genre_output_path, exist_ok=True)

        '''Get list of songs in the genre folder'''
        all_songs = os.listdir(genre_input_path)
        songs = []

        for s in all_songs:
            if os.path.isdir(os.path.join(genre_input_path, s)):
                songs.append(s)

        for song in tqdm(songs, desc=f"Processing {genre}"):

            song_input = os.path.join(genre_input_path, song)
            song_output = os.path.join(genre_output_path, song)
            os.makedirs(song_output, exist_ok=True)

            for stem in STEM_FILES:

                input_file = os.path.join(song_input, stem)
                output_file = os.path.join(song_output, stem.replace(".wav", ".npy"))

                '''Skip if already processed'''

                if os.path.exists(output_file):
                    continue

                '''Load audio with your Config settings (SR and DURATION)'''

                audio, _ = librosa.load(input_file, sr=SR, duration=DURATION)

                '''Compute Mel Spectrogram'''

                mel = librosa.feature.melspectrogram(
                    y=audio,
                    sr=SR,
                    n_mels=N_MELS,
                    n_fft=2048,
                    hop_length=512
                )

                '''Save the Mel Spectrogram'''

                np.save(output_file, mel.astype(np.float32))

In [8]:
# preprocess_dataset()

Processing pop: 100%|██████████| 100/100 [01:03<00:00,  1.58it/s]


In [9]:
def preprocess_noise():
    
    '''Create the output folder using your defined NOISE_OUTPUT'''
    os.makedirs(NOISE_OUTPUT, exist_ok=True)

    '''Get all wav files from NOISE_INPUT'''
    noise_files = []
    all_files = os.listdir(NOISE_INPUT)

    for f in all_files:
        if f.endswith(".wav"):
            noise_files.append(f)

    print(f"Processing {len(noise_files)} noise files")

    for file in tqdm(noise_files):

        input_path = os.path.join(NOISE_INPUT, file)
        output_path = os.path.join(NOISE_OUTPUT, file.replace(".wav", ".npy"))

        '''Skip if already processed'''

        if os.path.exists(output_path):
            continue

        '''Load audio using your SR (Sample Rate)'''

        audio, _ = librosa.load(input_path, sr=SR)

        '''Convert to Mel Spectrogram using your N_MELS'''

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=SR,
            n_mels=N_MELS,
            n_fft=2048,
            hop_length=512
        )

        '''Save the Mel Spectrogram'''

        np.save(output_path, mel.astype(np.float32))

In [10]:
# preprocess_noise()

Processing 2000 noise files


100%|██████████| 2000/2000 [00:54<00:00, 36.59it/s]


# Custom Dataset :

In [ ]:
class CustomDataset(Dataset):

    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):

        self.processed_dir = processed_dir
        self.song_dict = song_dict
        self.genres = genres
        self.size = size
        self.is_val = is_val

        self.genre_to_idx = {}

        for i in range(len(genres)):
            self.genre_to_idx[genres[i]] = i

        self.noise_files = []
        all_noise = os.listdir(noise_dir)

        for f in all_noise:
            if f.endswith(".npy"):
                self.noise_files.append(os.path.join(noise_dir, f))

        self.freq_mask = T.FrequencyMasking(freq_mask_param=60)
        self.time_mask = T.TimeMasking(time_mask_param=100)

        self.crop_size = 512


    def __len__(self):
        return self.size


    def __getitem__(self, idx):

        if self.is_val:
            genre = self.genres[idx % len(self.genres)]
        else:
            genre = random.choice(self.genres)

        songs = self.song_dict[genre]

        target_h = 128
        target_w = 1292

        mixed = np.zeros((target_h, target_w), dtype=np.float32)

        stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

        '''Stem Mixing'''
        for stem in stems:

            if not self.is_val and random.random() < 0.1:
                continue

            song = random.choice(songs)
            path = os.path.join(self.processed_dir, genre, song, stem)

            mel = np.load(path)

            if mel.shape[1] > target_w:
                mel = mel[:, :target_w]
            else:
                pad = target_w - mel.shape[1]
                mel = np.pad(mel, ((0,0),(0,pad)))

            gain = random.uniform(0.7, 1.3)

            mixed += mel * gain


        '''Time Shift'''
        if not self.is_val:
            shift = random.randint(-150, 150)
            mixed = np.roll(mixed, shift, axis=1)



        '''Multi Noise Addition'''
        if not self.is_val:

            noise_count = random.randint(1,3)

            for i in range(noise_count):

                noise_path = random.choice(self.noise_files)
                noise = np.load(noise_path)

                noise_len = noise.shape[1]

                if noise_len >= target_w:
                    noise = noise[:, :target_w]
                    start = 0
                else:
                    start = random.randint(0, target_w - noise_len)

                snr = random.uniform(8,25)

                signal_power = np.sqrt(np.mean(mixed**2)) + 1e-8
                noise_power = np.sqrt(np.mean(noise**2)) + 1e-8

                scale = signal_power / (10**(snr/10) * noise_power)

                mixed[:, start:start+noise.shape[1]] += noise * scale


        '''Random Crop'''
        if not self.is_val:

            if mixed.shape[1] > self.crop_size:
                start = random.randint(0, mixed.shape[1] - self.crop_size)
                mixed = mixed[:, start:start+self.crop_size]

        else:
            mixed = mixed[:, :self.crop_size]


        '''Loudness Augmentation'''
        if not self.is_val:
            loudness = random.uniform(0.7, 1.3)
            mixed = mixed * loudness


        '''Mel Normalization'''
        mel_db = librosa.power_to_db(np.maximum(mixed, 1e-10))

        mean = mel_db.mean()
        std = mel_db.std()

        mel_db = (mel_db - mean) / (std + 1e-6)

        mel = torch.tensor(mel_db).unsqueeze(0)


        '''Spec Augment'''
        if not self.is_val:

            mel = self.freq_mask(mel)
            mel = self.time_mask(mel)


        label = self.genre_to_idx[genre]

        return mel, label

# Train / Test Split

In [ ]:
def stratified_split(root_dir, val_ratio=0.15):

    train_songs = {}
    val_songs = {}

    # get genre folders
    genres = []

    for d in os.listdir(root_dir):
        path = os.path.join(root_dir, d)

        if os.path.isdir(path):
            genres.append(d)

    genres.sort()

    for genre in genres:

        genre_path = os.path.join(root_dir, genre)

        songs = []

        for s in os.listdir(genre_path):

            song_path = os.path.join(genre_path, s)

            if os.path.isdir(song_path):
                songs.append(s)

        songs.sort()
        random.shuffle(songs)

        # split into train and validation
        split = int(len(songs) * val_ratio)

        val_songs[genre] = songs[:split]
        train_songs[genre] = songs[split:]

        print("Genre", genre.ljust(10), "| Train:", len(train_songs[genre]), "| Val:", len(val_songs[genre]))

    return train_songs, val_songs, genres

In [ ]:
train_songs, val_songs, genres = stratified_split(GENRE_OUTPUT)

# Data Loaders

In [75]:
# train_dataset = CustomDataset(GENRE_OUTPUT, NOISE_OUTPUT, train_songs, genres, size=50000)
# val_dataset = CustomDataset(GENRE_OUTPUT, NOISE_OUTPUT, val_songs, genres, size=5000, is_val=True)


# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

# Model Architecture

## Scratch CNN

In [3]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        '''Convolutional Feature Extractor'''
        self.features = nn.Sequential(

            # ----- Block 1 -----
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   
            nn.BatchNorm2d(32),                           # Stabilizes training
            nn.ReLU(),                                    # Non-linear activation
            nn.MaxPool2d(2,2),                            # Downsample

            # ----- Block 2 -----
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # ----- Block 3 -----
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),                            

            # ----- Block 4 -----
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # ----- Block 5 -----
            nn.Conv2d(64, 128, kernel_size=3, padding=1), 
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),                           

            # ----- Block 6 -----
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # ----- Block 7 -----
            # High-level audio feature representation
            nn.Conv2d(128, 256, kernel_size=3, padding=1),# (256, 16, 16)
            nn.BatchNorm2d(256),
            nn.ReLU(),

            # Global Average Pooling
            # Reduces spatial dimension while keeping channel information
            nn.AdaptiveAvgPool2d((1,1))                   # (256, 1, 1)
        )


        '''Classifier Network'''
        self.classifier = nn.Sequential(

            nn.Flatten(),         # Convert (256,1,1) → (256)

            nn.Linear(256,128),   # Feature compression
            nn.ReLU(),
            nn.Dropout(0.5),      # Prevent overfitting

            nn.Linear(128,num_classes)  # Final genre prediction
        )


    def forward(self, x):

        '''Forward pass through feature extractor'''
        x = self.features(x)

        '''Forward pass through classifier'''
        x = self.classifier(x)

        return x

## CNN + LSTM

In [77]:
class CNN_LSTM(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        '''CNN Feature Extractor'''
        self.cnn = nn.Sequential(

            # ----- Block 1 -----
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # ----- Block 2 -----
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # ----- Block 3 -----
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # ----- Block 4 -----
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # ----- Block 5 -----
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )


        '''LSTM for Temporal Modeling'''
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )


        '''Classification Head'''
        self.fc = nn.Sequential(

            nn.Linear(256,128),   # BiLSTM output
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128,num_classes)
        )


    def forward(self, x):

        '''CNN feature extraction'''
        x = self.cnn(x)

        '''Collapse frequency dimension'''
        x = x.mean(dim=2)

        '''Prepare sequence for LSTM'''
        x = x.permute(0,2,1)


        '''Temporal modeling using LSTM'''
        x, _ = self.lstm(x)


        '''Temporal pooling'''
        x = x.mean(dim=1)


        '''Final classification'''
        x = self.fc(x)

        return x

In [4]:
model = CNN(num_classes=10).to(device)

# Loss Function, Optimizer, and Metrics

In [79]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scaler = torch.amp.GradScaler("cuda")

# Training Loop definition

In [80]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(loader)

    for x, y in pbar:

        # move data to device
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        # forward pass (mixed precision)
        with autocast(device_type="cuda"):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

        # update progress bar
        pbar.set_description(
            f"Train Loss {total_loss/(total/loader.batch_size):.4f} Acc {100*correct/total:.2f}%"
        )

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Validation Loop definition

In [81]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    # disable gradients during validation
    with torch.no_grad():

        for x, y in tqdm(loader):

            # move data to device
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1

# Model training and WndB logging

In [82]:
def train_model(model,train_loader,val_loader,criterion,optimizer,scheduler,scaler,device,epochs,patience):
    best_val_acc = 0
    patience_counter = 0

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        # TRAIN 
        train_loss, train_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            device
        )

        # VALIDATE
        val_loss, val_acc, val_f1 = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        print(
            f"\nEpoch [{epoch+1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Train Acc: {train_acc:.4f} "
            f"| Train F1: {train_f1:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.4f} "
            f"| Val F1: {val_f1:.4f}"
        )

        # SCHEDULER
        scheduler.step(val_acc)

        # WANDB LOG
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"]
        })

        # EARLY STOPPING
        if val_acc > best_val_acc:

            best_val_acc = val_acc
            patience_counter = 0

            torch.save(model.state_dict(), "best_model.pth")

            print("Model Improved. Saved.")

        # else:

        #     patience_counter += 1

        #     print(f"Early stopping counter: {patience_counter}/{patience}")

        #     if patience_counter >= patience:

        #         print("Early stopping triggered")
        #         break

    print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")

# Training execution¶

In [83]:
epochs= 20
patience= 5

In [84]:
# train_model(model=model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         criterion=criterion,
#         optimizer=optimizer,
#         scheduler=scheduler,
#         scaler=scaler,
#         device=device,
#         epochs=epochs,
#         patience=patience
#     )


Epoch 1/20


100%|██████████| 79/79 [00:11<00:00,  7.10it/s]



Epoch [1/20] | Train Loss: 1.2706 | Train Acc: 0.6793 | Train F1: 0.6733 | Val Loss: 1.4514 | Val Acc: 0.6802 | Val F1: 0.6680
Model Improved. Saved.

Epoch 2/20


100%|██████████| 79/79 [00:11<00:00,  6.99it/s]



Epoch [2/20] | Train Loss: 0.9353 | Train Acc: 0.8474 | Train F1: 0.8463 | Val Loss: 1.0428 | Val Acc: 0.7834 | Val F1: 0.7746
Model Improved. Saved.

Epoch 3/20


...


Epoch [18/20] | Train Loss: 0.6355 | Train Acc: 0.9795 | Train F1: 0.9795 | Val Loss: 0.7189 | Val Acc: 0.9240 | Val F1: 0.9260
Model Improved. Saved.

Epoch 19/20


100%|██████████| 79/79 [00:10<00:00,  7.43it/s]



Epoch [19/20] | Train Loss: 0.6327 | Train Acc: 0.9806 | Train F1: 0.9806 | Val Loss: 0.9869 | Val Acc: 0.8084 | Val F1: 0.8123

Epoch 20/20


100%|██████████| 79/79 [00:10<00:00,  7.34it/s]



Epoch [20/20] | Train Loss: 0.6274 | Train Acc: 0.9819 | Train F1: 0.9819 | Val Loss: 0.9522 | Val Acc: 0.8230 | Val F1: 0.8118

Best Validation Accuracy: 0.9240


# Model upload to kagglehub

In [85]:
# import kagglehub
# model_path = "/kaggle/working/Scratch_CNN.pth"


# torch.save(model.state_dict(), model_path)

# print("Model saved locally at:", model_path)

Model saved locally at: /kaggle/working/Scratch_CNN.pth


In [86]:
# KAGGLE_USERNAME = "uditmaurya1588"     # your Kaggle username
# MODEL = "Dlgenai"              # project/model name
# FRAMEWORK = "pytorch"                  # framework
# VARIATION = "Scratch_CNN-v3"              # version or hyperparameter variant

# handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

In [88]:
# kagglehub.model_upload(
#     handle,
#     model_path,
#     version_notes="Initial model version"
# )

Uploading Model https://api.kaggle.com/models/uditmaurya1588/Dlgenai/pytorch/Scratch_CNN-v3 ...
Starting upload for file /kaggle/working/Scratch_CNN.pth


Uploading: 100%|██████████| 2.49M/2.49M [00:00<00:00, 5.51MB/s]

Upload successful: /kaggle/working/Scratch_CNN.pth (2MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/uditmaurya1588/Dlgenai/pytorch/Scratch_CNN-v3


# Inference Code

In [20]:
import kagglehub
handle = "uditmaurya1588/dlgenai/pytorch/scratch_cnn-v3"
model_path = kagglehub.model_download(handle)
print("Downloaded:", model_path)

Downloaded: /kaggle/input/models/uditmaurya1588/dlgenai/pytorch/scratch_cnn-v3/2


# load 

In [22]:
model = CNN(num_classes=10).to(device)


model_file = '/kaggle/input/models/uditmaurya1588/dlgenai/pytorch/scratchcnn-v2/1/Scratch_CNN.pth'
state_dict = torch.load(model_file, map_location=device)

model.load_state_dict(state_dict)

print("Model loaded and ready for predictions.")

Model loaded and ready for predictions.


In [23]:
TARGET_WIDTH = 1292

# Test Dataset

In [25]:
class TestDataset(Dataset):

    def __init__(self, test_csv, mashup_dir):
        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir

    def __len__(self):
        return len(self.df)

    def audio_to_mel(self, path):

        # load audio
        audio, _ = librosa.load(path, sr=SR)

        # mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=SR,
            n_mels=N_MELS,
            n_fft=2048,
            hop_length=512
        )

        # crop / pad to fixed width
        if mel.shape[1] > TARGET_WIDTH:
            mel = mel[:, :TARGET_WIDTH]
        else:
            mel = np.pad(mel, ((0,0),(0, TARGET_WIDTH - mel.shape[1])))

        mel = librosa.power_to_db(np.maximum(mel, 1e-10))

        # normalize
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return mel.astype(np.float32)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = str(row["id"]).zfill(4)

        filename = row["filename"] if "filename" in row else sample_id + ".wav"

        path = os.path.join(self.mashup_dir, filename)

        mel = self.audio_to_mel(path)

        mel = torch.tensor(mel).unsqueeze(0)

        return mel, sample_id

In [26]:
test_csv = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

test_dataset = MashupTestDataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4   
)

In [12]:
genres = [
    "blues","classical","country","disco","hiphop",
    "jazz","metal","pop","reggae","rock"
]

predictions = []

model.eval()

with torch.no_grad():

    for mel, ids in tqdm(test_loader, desc="Predicting"):

        mel = mel.to(device)

        outputs = model(mel)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for sample_id, pred in zip(ids, preds):
            predictions.append((sample_id, genres[pred]))

Predicting: 100%|██████████| 189/189 [03:11<00:00,  1.01s/it]


In [13]:
import pandas as pd

submission = pd.DataFrame(predictions, columns=["id", "genre"])

submission.to_csv("submission.csv", index=False)


In [14]:
submission

,id,genre
0,0001,pop
1,0002,classical
2,0003,disco
3,0004,metal
4,0005,country
...,...,...
3015,3016,rock
3016,3017,jazz
3017,3018,reggae
3018,3019,country
